# HEADING

In [44]:
from haystack import Pipeline, Document, component
from milvus_haystack import MilvusDocumentStore
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter
from haystack.utils import Secret
from pymilvus import MilvusClient, DataType
from typing import List
from pymilvus import MilvusClient, DataType
from dotenv import load_dotenv
import os
from openai import AsyncOpenAI
import asyncio

# load_dotenv(override=True)
load_dotenv()

True

In [5]:
client = MilvusClient(
    uri=os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.getenv("ZILLIZ_CLUSTER_TOKEN")
)

In [6]:
col_name = client.list_collections()[0]
client.load_collection(col_name)
client.get_load_state(col_name)

{'state': <LoadState: Loaded>}

In [7]:
filter = 'metadata["tags"] == []'
# filter = ""
# filter = 'metadata["page_number"] == 6'

res = client.query(
    collection_name=col_name,
    filter=filter,
    output_fields=["id", "text"],
    # limit=2
)

print(res)
print(len(res))

data: ["{'id': '00d10142bdb9d200125ec87d0a2dab835cc4f23a89fa318ee0acbb187fb9326d', 'text': 'with the 1st project, then we willput the student in the pool for consideration ■ Ifit’sa really urgent request from a client then Beam Data may override the rules by selecting the student who has specific skills required for the project - E.g., ifthere’s only one student knows Elasticsearch database then we may choose that student for help ○ How ready isthe student for jobs? ■ Ifthe student has not accomplished much on building the resume and portfolio, then we’d favor the student less because the faculty may want the student to focus on job search ● Exceptions will be '}", "{'id': '049ca25d41fdad9880c104633164071b5321a320f1791c657a7aa4375a651b19', 'text': 'data Holding out part of the Training Test\\ndata for assessing the\\nmodel Full: Cross-validation Test ● Model training ● Variance assessment and hyperparameter optimization ● Testing the model Python Fundamentals | Data Science Track | www

In [8]:
def dict_to_doc(d: dict) -> Document:
    return Document(id=d["id"], embedding=d["vector"], content=d["text"], meta={"metadata": res[0]["metadata"]})


In [ ]:
res = client.query(
    collection_name=col_name,
    filter=filter,
    output_fields=["*"],
)

# docs = [Document.from_dict(d) for d in res] # this doesn't work
# res[0]["metadata"]
docs = [dict_to_doc(d) for d in res]
docs[0]

Document(id=00d10142bdb9d200125ec87d0a2dab835cc4f23a89fa318ee0acbb187fb9326d, content: 'with the 1st project, then we willput the student in the pool for consideration ■ Ifit’sa really urg...', meta: {'metadata': {'tags': [], 'source_id': '8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e', 'file_path': 'Project Management Requirements Handbook.pdf', 'page_number': 6, 'split_overlap_ids': ['de0c92f66393af2edab3068f6495974c41fc6f3fc92af9f451d4d3e9b31a0703', '87c06980c235ebf511449e430bce7afcbc0ab48b51cdd8fd78da9beec35d9541']}}, embedding: vector of size 768)

In [ ]:
# @component
# class LLMTagger:
#     def __init__(self, prompt: str):
#         self.prompt_template = prompt

#     @component.output_types(documents=List[Document])
#     def run(self, documents: List[Document]):
#         docs = [self._add_metadata(doc) for doc in documents]
#         return {"documents": documents}
    
#     async def tag(self, documents: List[Document]):
#         tasks = []
#         asyncio.gather(*tasks)
#         asyncio.run()

In [12]:
res = client.get(
    collection_name=col_name,
    ids=["00d10142bdb9d200125ec87d0a2dab835cc4f23a89fa318ee0acbb187fb9326d",
         "049ca25d41fdad9880c104633164071b5321a320f1791c657a7aa4375a651b19"],
    # output_fields=["id", "text"]
    output_fields = ["*"]
)

print(len(res))
print(res)

2
data: ["{'id': '00d10142bdb9d200125ec87d0a2dab835cc4f23a89fa318ee0acbb187fb9326d', 'vector': [np.float32(0.023748618), np.float32(-0.030406652), np.float32(-0.043143496), np.float32(-0.02374954), np.float32(-0.019217484), np.float32(-0.020491764), np.float32(-0.002154597), np.float32(-0.07080215), np.float32(-0.019675087), np.float32(0.008141645), np.float32(0.07105464), np.float32(-0.021231893), np.float32(-0.03384295), np.float32(0.06446489), np.float32(-0.036160856), np.float32(-0.03861485), np.float32(-0.01008533), np.float32(-0.007533311), np.float32(-0.028022202), np.float32(-0.011312209), np.float32(-0.011124962), np.float32(-0.01676099), np.float32(-0.0419714), np.float32(0.005551516), np.float32(-0.0011390364), np.float32(0.026860863), np.float32(0.0377212), np.float32(0.059633028), np.float32(-0.004930998), np.float32(-0.0038954413), np.float32(0.039896403), np.float32(0.019453345), np.float32(-0.033872243), np.float32(-0.030432243), np.float32(1.6874037e-06), np.float32(-0

# Custom components
## MilvusQueryRetriever

In [17]:
@component
class MilvusQueryRetriever:
    def __init__(self):
        self.client = MilvusClient(
            uri=Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
            token=Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value()
        )

    @component.output_types(documents=List[Document])
    def run(self, collection_name: str, ids: list[str] | None=None):
        if ids:
            # Retrieve specific IDs
            res = self.client.get(
                collection_name=col_name,
                ids=ids,
                output_fields = ["*"]
            )
        else:
            # If no IDs provided, query all entities with empty tags
            res = self.client.query(
                collection_name=collection_name,
                filter='metadata["tags"] == []',
                output_fields=["*"],
            )

        # Convert from entity format to Document format
        return [self.dict_to_doc(d) for d in res]
    
    def dict_to_doc(self, d: dict) -> Document:
        return Document(id=d["id"], embedding=d["vector"], content=d["text"], meta={"metadata": res[0]["metadata"]})

In [18]:
query_retriever = MilvusQueryRetriever()

In [25]:
docs = query_retriever.run(collection_name=col_name)
print(len(docs))
docs[0]

70


Document(id=00d10142bdb9d200125ec87d0a2dab835cc4f23a89fa318ee0acbb187fb9326d, content: 'with the 1st project, then we willput the student in the pool for consideration ■ Ifit’sa really urg...', meta: {'metadata': {'tags': [], 'source_id': '8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e', 'file_path': 'Project Management Requirements Handbook.pdf', 'page_number': 6, 'split_overlap_ids': ['de0c92f66393af2edab3068f6495974c41fc6f3fc92af9f451d4d3e9b31a0703', '87c06980c235ebf511449e430bce7afcbc0ab48b51cdd8fd78da9beec35d9541']}}, embedding: vector of size 768)

In [28]:
ids = ["00d10142bdb9d200125ec87d0a2dab835cc4f23a89fa318ee0acbb187fb9326d",
         "049ca25d41fdad9880c104633164071b5321a320f1791c657a7aa4375a651b19"]
docs = query_retriever.run(collection_name=col_name, ids=ids)
print(len(docs))
docs[1]

2


Document(id=049ca25d41fdad9880c104633164071b5321a320f1791c657a7aa4375a651b19, content: 'data Holding out part of the Training Test
data for assessing the
model Full: Cross-validation Test ...', meta: {'metadata': {'tags': [], 'source_id': '8ac81db781a7877b09869dd6be3c0fdd7a4a737864c9f647bb3cfe3af66cb71e', 'file_path': 'Project Management Requirements Handbook.pdf', 'page_number': 6, 'split_overlap_ids': ['de0c92f66393af2edab3068f6495974c41fc6f3fc92af9f451d4d3e9b31a0703', '87c06980c235ebf511449e430bce7afcbc0ab48b51cdd8fd78da9beec35d9541']}}, embedding: vector of size 768)

## SynchronousLLMTagger

In [37]:
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator

prompt_template = """Given the context inside the triple backticks below, tag the topic. You may tag multiple topics.
                     Please format the tags as strings in a Python list, e.g. ["current events", "USA"].
                     Return only the list; return no other text.

                     Context:
                     ```
                     {{doc.content}}
                     ```

                     Tags as a Python list of strings:
                  """

pipe = Pipeline()
pipe.add_component("prompt_builder", PromptBuilder(template=prompt_template))
pipe.add_component("generator", OpenAIGenerator(model="gpt-3.5-turbo", generation_kwargs={"temperature": 0.7, "max_tokens": 500}))
pipe.connect("prompt_builder", "generator")

🚅 Components
  - prompt_builder: PromptBuilder
  - generator: OpenAIGenerator
🛤️ Connections
  - prompt_builder.prompt -> generator.prompt (str)

In [ ]:
Secret.from_env_var("OPENAI_API_KEY").resolve_value()

In [ ]:
pipe.get_component("generator").api_key.resolve_value()

In [38]:
doc = Document(content="Lebron James is the career points leader in the NBA.")

tag = pipe.run({"prompt_builder": {"doc": doc}})
tag

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Project `proj_8GIGLN7dEjlzdYg3mrJzoEVv` does not have access to model `gpt-3.5-turbo`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [ ]:
@component
class SyncLLMTagger:
    def __init__(self, prompt_template: str):
        pipe = Pipeline()
        pipe.add_component("prompt_builder", PromptBuilder(template=prompt_template))
        pipe.add_component("generator", OpenAIGenerator(generation_kwargs={"temperature": 0.7, "max_tokens": 500}))
        pipe.connect("prompt_builder", "generator")
        self.tag_pipe = pipe

    @component.output_types(tags=list[list[str]])
    def run(self, documents: List[Document]):
        tags = []
        for doc in documents:
            tag = self.tag_pipe.run({"prompt_builder": {"doc": doc}}) # This is a string though
            tags.append(tag)
        
        return {"tags": tags}

## Upsert